In [1]:
import requests
import json
import mujoco

repo_id = "lerobot/full_folding"
mjcf_path = "../arms/openarm_mujoco/v1/scene.xml"

# 1. Download the dataset's configuration file directly from the HF Hub
config_url = f"https://huggingface.co/datasets/{repo_id}/raw/main/meta/info.json"

response = requests.get(config_url)
if response.status_code == 200:
    hf_config = json.loads(response.text)
    # Extract the exact string names ordered by their vector index
    dataset_joint_names = hf_config['features']['observation.state']['names']
else:
    # Fallback default names for standard OpenArm 2.0 configurations if download fails
    dataset_joint_names = [f"index_{i}" for i in range(14)]

# 2. Load your MuJoCo model
model = mujoco.MjModel.from_xml_path(mjcf_path)

print(f"Dataset Vector Length: {len(dataset_joint_names)}")
print(f"MuJoCo QPos Length: {model.nq}\n")

print(f"{'Index':<6} | {'Dataset Joint Name':<35} | {'MuJoCo QPos Target':<30}")
print("-" * 80)

# Print out the mapping alignment
max_lines = max(len(dataset_joint_names), model.nq)
for i in range(max_lines):
    ds_name = dataset_joint_names[i] if i < len(dataset_joint_names) else "---"

    # Identify what MuJoCo is hosting at this specific qpos address
    mj_name = "---"
    for j in range(model.njnt):
        if model.jnt_qposadr[j] <= i < (model.jnt_qposadr[j] + (7 if model.jnt_type[j]==0 else 1)):
            j_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, j)
            j_type = ["free", "ball", "slide", "hinge"][model.jnt_type[j]]
            mj_name = f"{j_name} ({j_type})"
            break

    print(f"{i:<6} | {ds_name:<35} | {mj_name:<30}")

Dataset Vector Length: 16
MuJoCo QPos Length: 18

Index  | Dataset Joint Name                  | MuJoCo QPos Target            
--------------------------------------------------------------------------------
0      | right_joint_1.pos                   | openarm_left_joint1 (hinge)   
1      | right_joint_2.pos                   | openarm_left_joint2 (hinge)   
2      | right_joint_3.pos                   | openarm_left_joint3 (hinge)   
3      | right_joint_4.pos                   | openarm_left_joint4 (hinge)   
4      | right_joint_5.pos                   | openarm_left_joint5 (hinge)   
5      | right_joint_6.pos                   | openarm_left_joint6 (hinge)   
6      | right_joint_7.pos                   | openarm_left_joint7 (hinge)   
7      | right_gripper.pos                   | openarm_left_finger_joint1 (slide)
8      | left_joint_1.pos                    | openarm_left_finger_joint2 (slide)
9      | left_joint_2.pos                    | openarm_right_joint1 (hinge)  
10 